# 07 — Solar components and energy closure

**Learning objectives:** distinguish GHI, DNI, DHI, and direct-horizontal irradiance; evaluate midpoint geometry; verify physical closure; and compare four primary solar paths. Automatic compilation also supports ERA5 and mixed sources.

**Network:** none. **Expected runtime:** under one minute. Values are intentionally synthetic so each branch is controlled.

On a horizontal surface, `GHI = DHI + DNI × cos(zenith)`. DNI describes a surface normal to the solar beam; multiplying by the zenith cosine projects it horizontally. Near the horizon, dividing a small horizontal beam by a tiny cosine is unstable, so the current pipeline assigns GHI to diffuse radiation below its approximately 5° elevation floor, even when source DNI is supplied. This approximation can suppress real low-sun direct gains, especially in northern winter. A zero horizontal closure residual does not independently validate the solar partition.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from smhi2epw import processing, solar

LATITUDE, LONGITUDE = 57.7156, 11.9924
index = pd.date_range("2023-06-21", periods=24, freq="h", tz="UTC")
midpoints = index - pd.Timedelta(minutes=30)
zenith, cos_zenith = solar.solar_zenith(midpoints, LATITUDE, LONGITUDE)
etrh, etrn = solar.extraterrestrial_radiation(midpoints, cos_zenith)
daylight = cos_zenith > 0.087
ghi = np.where(daylight, 700 * np.clip(cos_zenith, 0, None), 0.0)
beam_horizontal = 0.65 * ghi
dni = np.where(daylight, beam_horizontal / np.clip(cos_zenith, 0.087, None), 0.0)

In [ ]:
inputs = {
    "strang": pd.DataFrame(
        {"ghi": ghi, "dni": dni, "dirh": beam_horizontal}, index=index
    ),
    "measured+strang_partition": pd.DataFrame(
        {"ghi": ghi, "dni": dni, "dirh": beam_horizontal, "ghi_measured": 0.9 * ghi},
        index=index,
    ),
    "strang_ghi+erbs": pd.DataFrame({"ghi": ghi}, index=index),
    "measured+erbs": pd.DataFrame({"ghi": ghi, "ghi_measured": 0.9 * ghi}, index=index),
}
rows = []
outputs = {}
for intended, frame in inputs.items():
    report = processing.ProcessingReport()
    processing.apply_solar(frame, LATITUDE, LONGITUDE, report)
    outputs[intended] = frame
    rows.append(
        {
            "intended_path": intended,
            "reported_path": report.solar_source,
            "max_residual_W_m2": report.energy_balance_max_residual,
            "clamped_hours": report.clamped_dni_hours,
        }
    )
pd.DataFrame(rows).set_index("intended_path")

In [ ]:
frame = outputs["measured+strang_partition"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(index, frame.ghi, label="GHI")
ax.plot(index, frame.dhi, label="DHI")
ax.plot(index, frame.dni, label="DNI")
ax.set(ylabel="Irradiance (W/m²)", xlabel="UTC hour label")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()

In [ ]:
residual = frame.ghi.to_numpy() - (
    frame.dhi.to_numpy() + frame.dni.to_numpy() * np.clip(cos_zenith, 0, None)
)
float(np.max(np.abs(residual)))

## Key takeaways and exercise

Measured GHI changes the total energy; STRÅNG or Erbs supplies a partition. Physical caps are followed by a DHI readjustment so closure is not sacrificed.

**Try it:** multiply measured GHI by 1.5 while leaving the STRÅNG model unchanged. Which component changes most, and when does the extraterrestrial DNI cap become active?